In [1]:
import re
from pathlib import Path

import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

/home/abdelaziz/Documents/Arix-Research-Paper-Recommendation-System/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Locate the repo from the current directory, so the notebook runs from anywhere.
HERE = Path.cwd()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError(f"Start the notebook inside the repo: no pyproject.toml above {HERE}.")
DATA_DIR = ROOT / "data"

In [3]:
def ensure_nltk(pkg):
    """Download an NLTK package only when it is missing; fail with context if that fails."""
    for name in (f"corpora/{pkg}", f"corpora/{pkg}.zip"):
        try:
            nltk.data.find(name)
            return
        except LookupError:
            pass
    try:
        nltk.download(pkg, raise_on_error=True)
    except Exception as e:
        raise RuntimeError(
            f"NLTK data '{pkg}' is missing and could not be downloaded. "
            "Check your network or pre-install it (see nltk.data.path)."
        ) from e

In [4]:
for _pkg in ("stopwords", "wordnet"):
    ensure_nltk(_pkg)


In [5]:
df = pd.read_csv(DATA_DIR / "arxiv_papers_v2.csv")

In [6]:
df.head()

,request_id,retrieved_at,domain,arxiv_id,title,summary,authors,primary_category,categories,published,updated,abs_url,pdf_url
0,https://arxiv.org/api/Nn9ZAl7r/hPtw2SqI662/Ri9DWc,2026-10-30T05:02:17+00:00,cs,2610.40655v1,Elucidating the Design Space of LM Data Attrib...,Data attribution methods for Language Models (...,Steven Yu; Sheng-Yu Wang; Kimia Hamidieh; Anto...,cs.CL,cs.CL,2026-10-29T17:59:53Z,2026-10-29T17:59:53Z,http://arxiv.org/abs/2610.40655v1,https://arxiv.org/pdf/2610.40655v1
1,https://arxiv.org/api/Nn9ZAl7r/hPtw2SqI662/Ri9DWc,2026-10-30T05:02:17+00:00,cs,2610.40654v1,Propose-Filter-Release: Scaling Noise to Typic...,An estimator can be accurate and stable on typ...,Moshe Shenfeld; Eliad Tsfadia; Katrina Ligett,cs.LG,cs.LG,2026-10-29T17:59:49Z,2026-10-29T17:59:49Z,http://arxiv.org/abs/2610.40654v1,https://arxiv.org/pdf/2610.40654v1
2,https://arxiv.org/api/Nn9ZAl7r/hPtw2SqI662/Ri9DWc,2026-10-30T05:02:17+00:00,cs,2610.40653v1,Micro-Edits: Computational Support for Fine-Sc...,"In film editing, every frame matters. While ma...",Nhan Tran; Hijung Valentina Shin; Abe Davis; M...,cs.HC,cs.HC; cs.CV,2026-10-29T17:59:49Z,2026-10-29T17:59:49Z,http://arxiv.org/abs/2610.40653v1,https://arxiv.org/pdf/2610.40653v1
3,https://arxiv.org/api/Nn9ZAl7r/hPtw2SqI662/Ri9DWc,2026-10-30T05:02:17+00:00,cs,2610.40649v1,Rethinking Neural Thickets Without Weight Pert...,The Neural Thickets hypothesis posits that pre...,Nicolas Dufour; Elouan Gardès; Amil Dravid; Pa...,cs.LG,cs.LG; cs.AI; cs.CV,2026-10-29T17:59:29Z,2026-10-29T17:59:29Z,http://arxiv.org/abs/2610.40649v1,https://arxiv.org/pdf/2610.40649v1
4,https://arxiv.org/api/Nn9ZAl7r/hPtw2SqI662/Ri9DWc,2026-10-30T05:02:17+00:00,cs,2610.40647v1,"Read, Allocate, Route: Role-Coupled Tokenizati...",Pixel-space generative Transformers typically ...,Jiongfan Yang; Weichen Fan; Haiwen Diao; Zhong...,cs.CV,cs.CV; cs.AI; cs.LG; cs.NE,2026-10-29T17:59:21Z,2026-10-29T17:59:21Z,http://arxiv.org/abs/2610.40647v1,https://arxiv.org/pdf/2610.40647v1


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   request_id        2000 non-null   str  
 1   retrieved_at      2000 non-null   str  
 2   domain            2000 non-null   str  
 3   arxiv_id          2000 non-null   str  
 4   title             2000 non-null   str  
 5   summary           2000 non-null   str  
 6   authors           2000 non-null   str  
 7   primary_category  2000 non-null   str  
 8   categories        2000 non-null   str  
 9   published         2000 non-null   str  
 10  updated           2000 non-null   str  
 11  abs_url           2000 non-null   str  
 12  pdf_url           2000 non-null   str  
dtypes: str(13)
memory usage: 203.3 KB


In [8]:
df["text"] = df["title"].str.strip() + ". " + df["summary"].str.strip()

In [9]:
### Dataset Preprocessing
wnl = WordNetLemmatizer()
def preprocessing(text):
    text_clean = re.sub('[^a-zA-Z]', ' ', text)
    text_clean= text_clean.lower()
    text_clean = text_clean.split()

    sw = set(stopwords.words("english"))

    text_clean = [wnl.lemmatize(word) for word in text_clean if not word in sw]
    text_final = " ".join(text_clean)
    return text_final

In [10]:
tqdm.pandas()

df["text_clean"] = df["text"].progress_apply(preprocessing)

100%|██████████| 2000/2000 [00:04<00:00, 481.44it/s] 


In [11]:
model = SentenceTransformer("BAAI/bge-base-en-v1.5")
df["text_clean_embeddings"] = df["text_clean"].progress_apply(model.encode)

100%|██████████| 2000/2000 [06:56<00:00,  4.80it/s]


In [12]:
df.to_csv(DATA_DIR / "arxiv_papers_preprocessed_v2.csv", index=False)